# Notebook 07 - Evaluate KIRNEWS Classification

This notebook evaluates whether each model can classify Kirundi news articles from KIRNEWS into English category labels.

This is still a proxy task. It is useful because KIRNEWS is labeled, but classification accuracy does not prove broad language quality.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from kirundi_sft_starter.data import load_kirnews_prompts
from kirundi_sft_starter.evals import normalize_label, score_kirnews_predictions
from kirundi_sft_starter.utils import ensure_dir, load_yaml, read_jsonl

project = load_yaml(ROOT / "configs/project.yaml")
eval_cfg = load_yaml(ROOT / "configs/evaluation.yaml")
labels = eval_cfg["kirnews_classification"]["labels"]
labels

## Prepare KIRNEWS prompts

The prompt asks for one English label only. The default label set uses the labels from the dataset card, including `history`, `tourism`, and `fashion`.

In [ ]:
kirnews = load_kirnews_prompts(project)
kirnews[["prompt_id", "gold_label", "title"]].head()

## Generate predictions

Run these from the repo root. Add `--dry-run` if you only want placeholder files.

```bash
python scripts/generate_model_responses.py --model-key base --prompts data/eval/kirnews_prompts.jsonl --output results/responses/kirnews_base.jsonl
python scripts/generate_model_responses.py --model-key sft_raw --prompts data/eval/kirnews_prompts.jsonl --output results/responses/kirnews_sft_raw.jsonl
python scripts/generate_model_responses.py --model-key sft_adapted --prompts data/eval/kirnews_prompts.jsonl --output results/responses/kirnews_sft_adapted.jsonl
```

In [ ]:
prediction_files = {
    "base": ROOT / "results/responses/kirnews_base.jsonl",
    "sft_raw": ROOT / "results/responses/kirnews_sft_raw.jsonl",
    "sft_adapted": ROOT / "results/responses/kirnews_sft_adapted.jsonl",
}

frames = []
for model, path in prediction_files.items():
    if path.exists():
        df = pd.DataFrame(read_jsonl(path))
        df["model"] = model
        df = df.rename(columns={"response": "prediction"})
        frames.append(df)

if not frames:
    raise FileNotFoundError("No KIRNEWS prediction files found. Generate predictions first.")

predictions = pd.concat(frames, ignore_index=True)
predictions[["model", "prompt_id", "gold_label", "prediction"]].head()

In [ ]:
rows = []
for model, group in predictions.groupby("model"):
    scores = score_kirnews_predictions(group, labels)
    rows.append({
        "model": model,
        "accuracy": round(scores["accuracy"], 4),
        "macro_f1": round(scores["macro_f1"], 4),
        "num_examples": len(group),
        "notes": "proxy automatic classification eval",
    })

summary = pd.DataFrame(rows)
display(summary)

output_path = ROOT / eval_cfg["kirnews_classification"]["output_path"]
ensure_dir(output_path)
summary.to_csv(output_path, index=False)
print("Saved", output_path)

## Suggested qualitative table

After scoring, inspect examples where `sft_adapted` differs from `sft_raw`. Those rows are the most useful for understanding whether Adaption changed the downstream behavior or merely changed surface form.